# GR00T N1.7 — Step 2: SageMaker Training Job

This notebook launches a SageMaker training job to fine-tune GR00T N1.7-3B on the BridgeData V2 dataset prepared in notebook 01.

**Prerequisites:**
- Run `01_data_preparation.ipynb` first (dataset must be in S3)
- HuggingFace token with access to `nvidia/GR00T-N1.7-3B`
- Accept the model license at https://huggingface.co/nvidia/GR00T-N1.7-3B

**Instance:** ml.g6e.48xlarge (8× L40S 48GB) or ml.p4d.24xlarge (8× A100 40GB)

**Training time:** ~4-8 hours for 2,000 steps

## 1. Setup

In [ ]:
# Install/upgrade SageMaker SDK if needed
%pip install -Uq "sagemaker==3.8.0"

### Authentication Setup

Configure Hugging Face authentication. The token is passed to the training container so it can download the GR00T model.

In [ ]:
from getpass import getpass
from huggingface_hub import login

# Prompt for Hugging Face token (input is hidden)
hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

### SageMaker Session Configuration

Initialize SageMaker session for distributed training and model management.

**IAM Permissions:** The SageMaker execution role used for training should follow the principle of least privilege. At minimum, it needs:
- `s3:GetObject` and `s3:PutObject` scoped to the specific training data and output bucket paths
- `sagemaker:CreateTrainingJob` for launching training jobs

In [ ]:
import os
import boto3
import sagemaker
from sagemaker.core.helper import session_helper

sagemaker_session = session_helper.Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Bucket: {bucket_name}")

## 2. Training Configuration

In [ ]:
from sagemaker.core.training.configs import (
    SourceCode, Compute, InputData, OutputDataConfig,
    StoppingCondition, CheckpointConfig,
)
from sagemaker.train import ModelTrainer

# --- Instance & image ---
instance_type = "ml.g6e.48xlarge"   # 8× L40S 48GB
instance_count = 1                   # Single node (8 GPUs)
image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.7-gpu-py312"

# --- Training hyperparameters ---
MAX_STEPS = 2000
BATCH_SIZE = 32
SAVE_STEPS = 2000

# --- S3 paths ---
if default_prefix:
    S3_PREFIX = f"{default_prefix}/groot-finetuning/datasets/bridge_lerobot"
else:
    S3_PREFIX = "groot-finetuning/datasets/bridge_lerobot"

dataset_s3_uri = f"s3://{bucket_name}/{S3_PREFIX}"

job_name = "groot-n17-finetune-bridge"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/groot-finetuning/{job_name}"
else:
    output_path = f"s3://{bucket_name}/groot-finetuning/{job_name}"

print(f"Instance: {instance_type} × {instance_count}")
print(f"Dataset: {dataset_s3_uri}")
print(f"Output: {output_path}")
print(f"Steps: {MAX_STEPS}, Batch: {BATCH_SIZE}")

## 3. Create ModelTrainer

In [ ]:
# Environment variables passed to the training container
env = {
    "HF_TOKEN": hf_token,
    "MAX_STEPS": str(MAX_STEPS),
    "BATCH_SIZE": str(BATCH_SIZE),
    "SAVE_STEPS": str(SAVE_STEPS),
    "FI_PROVIDER": "efa",                    # Elastic Fabric Adapter for high-performance networking
    "NCCL_PROTO": "simple",                  # NCCL (NVIDIA Collective Communications Library) protocol for multi-GPU communication
    "NCCL_SOCKET_IFNAME": "eth0",            # Network interface
    "NCCL_IB_DISABLE": "1",                  # Disable InfiniBand
    "NCCL_DEBUG": "DEBUG",                    # NCCL debug level
    "NCCL_TIMEOUT": str(3600),
    "TORCH_NCCL_HEARTBEAT_TIMEOUT_SEC": str(3600),
    "TORCH_NCCL_ENABLE_MONITORING": str(0)
}

# Source code — the scripts/ folder is uploaded to the container
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    entry_script="run_finetuning.sh",
)

# Compute
compute_configs = Compute(
    instance_type=instance_type,
    instance_count=instance_count,
    keep_alive_period_in_seconds=3600,
)

# Create trainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    environment=env,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=36000),  # 10 hours max
    output_data_config=OutputDataConfig(s3_output_path=output_path),
    checkpoint_config=CheckpointConfig(
        s3_uri=output_path + "/checkpoints",
        local_path="/opt/ml/checkpoints",
    ),
)

print("ModelTrainer configured.")

## 4. Configure Input Data

In [ ]:
# The training dataset is downloaded from S3 to /opt/ml/input/data/train/ on the instance
train_input = InputData(
    channel_name="train",
    data_source=dataset_s3_uri,
)

data = [train_input]

print(f"Input channel 'train': {dataset_s3_uri}")

## 5. Launch Training Job

In [ ]:
print(f"Launching training job: {job_name}")
print(f"Instance: {instance_type} × {instance_count}")
print(f"Expected duration: ~4-8 hours")
print()

model_trainer.train(input_data_config=data, wait=False)

print("\nTraining job submitted.")
print("Monitor in SageMaker Console → Training Jobs")
print(f"Output will be at: {output_path}")

## 6. (Optional) Wait for Job & Check Status

In [ ]:
# Uncomment to wait for the job to complete (blocks the notebook)
# model_trainer.latest_training_job.wait(logs="All")

# Check job status
sm_client = boto3.client("sagemaker")

# Find latest job with our prefix
response = sm_client.list_training_jobs(
    NameContains=job_name,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)

for job in response["TrainingJobSummaries"]:
    print(f"{job['TrainingJobName']}: {job['TrainingJobStatus']}")

## 7. Download Model Artifacts (after training completes)

Once the job is `Completed`, download the fine-tuned checkpoint for evaluation.

In [ ]:
import tarfile
import boto3, os

sm_client = boto3.client('sagemaker')
job_name = 'groot-n17-finetune-bridge'

# Find the completed job
response = sm_client.list_training_jobs(
    NameContains=job_name,
    StatusEquals='Completed',
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1,
)

if not response['TrainingJobSummaries']:
    print('No completed training jobs found via status filter. Trying without filter...')
    response = sm_client.list_training_jobs(
        NameContains=job_name, SortBy='CreationTime', SortOrder='Descending', MaxResults=5)
    for job in response['TrainingJobSummaries']:
        print(f"  {job['TrainingJobName']}: {job['TrainingJobStatus']}")

if not response['TrainingJobSummaries'] or response['TrainingJobSummaries'][0]['TrainingJobStatus'] != 'Completed':
    print('No completed job to download.')
else:
    completed_job = response['TrainingJobSummaries'][0]['TrainingJobName']
    job_desc = sm_client.describe_training_job(TrainingJobName=completed_job)
    model_s3_uri = job_desc['ModelArtifacts']['S3ModelArtifacts']
    
    print(f'Job: {completed_job}')
    print(f'Model artifacts: {model_s3_uri}')
    
    # Download
    local_tar = f'./model_artifacts/{completed_job}/model.tar.gz'
    local_model_dir = f'./model_artifacts/{completed_job}/extracted/'
    os.makedirs(os.path.dirname(local_tar), exist_ok=True)
    os.makedirs(local_model_dir, exist_ok=True)
    
    # Parse S3 URI
    s3_parts = model_s3_uri.replace('s3://', '').split('/', 1)
    s3_bucket, s3_key = s3_parts[0], s3_parts[1]
    
    print('Downloading...')
    boto3.client('s3').download_file(s3_bucket, s3_key, local_tar)
    
    print('Extracting...')
    with tarfile.open(local_tar, 'r:gz') as tar:
        tar.extractall(path=local_model_dir)
    
    print(f'Model extracted to: {local_model_dir}')
    print('\nYou can now run 03_evaluation.ipynb')